# Workshop 5 Live Coding Demo: Control Flow

This notebook is for the live coding sections of Workshop 5.

We add:

```text
Comparison
IfStatement
WhileStatement
```

Core idea:

```text
Control flow = the evaluator deciding what executes next
```

# Part 0 — Setup

In [68]:
from dataclasses import dataclass
from typing import Any

In [6]:
@dataclass
class Number:
    value: int

@dataclass
class Boolean:
    value: bool

@dataclass
class Variable:
    name: str

@dataclass
class Assignment:
    name: str
    value: Any

@dataclass
class BinaryOp:
    op: str
    left: Any
    right: Any

# NEW NODES!
@dataclass
class Comparison:
    op: str
    left: Any
    right: Any

@dataclass
class IfStatement:
    condition: Any
    body: Any

@dataclass
class WhileStatement:
    condition: Any
    body: Any

In [7]:
class Environment:
    def __init__(self):
        self.values = {}

    def define(self, name, value):
        self.values[name] = value
        return value

    def get(self, name):
        if name not in self.values:
            raise NameError(f"{name!r} is not defined")
        return self.values[name]

    def __repr__(self):
        return f"Environment({self.values})"

# Part 1 — Existing Evaluator

This supports:

```text
Number
Boolean
BinaryOp
Variable
Assignment
```

It does **not** yet support:

```text
Comparison
IfStatement
WhileStatement
```

In [75]:
def evaluate_base(node, env):
    if isinstance(node, Number):
        return node.value

    if isinstance(node, Boolean):
        return node.value

    if isinstance(node, BinaryOp):
        left = evaluate_base(node.left, env)
        right = evaluate_base(node.right, env)

        if node.op == "+":
            return left + right
        if node.op == "-":
            return left - right
        if node.op == "*":
            return left * right
        if node.op == "/":
            return left / right

        raise ValueError(f"Unknown operator: {node.op}")

    if isinstance(node, Variable):
        return env.get(node.name)

    if isinstance(node, Assignment):
        value = evaluate_base(node.value, env)
        env.define(node.name, value)
        return value

    if isinstance(node, Comparison):
        left = evaluate_base(node.left, env)
        right = evaluate_base(node.left, env)

        if node.op == ">":
            return left > right
        if node.op == "<":
            return left < right
        if node.op == "==":
            return left == right
        

    raise TypeError(f"Unknown node: {node}")

In [70]:
env = Environment()
evaluate_base(Assignment("x", Number(5)), env)
evaluate_base(BinaryOp("+", Variable("x"), Number(2)), env)

7

# Part 2 — Live Coding: Add Comparisons

Prompt:

> What should `x > 3` evaluate to?

```

In [78]:
# Starter: add the Comparison case

def evaluate(node, env):
    if isinstance(node, Number):
        return node.value

    if isinstance(node, Boolean):
        return node.value

    if isinstance(node, BinaryOp):
        left = evaluate(node.left, env)
        right = evaluate(node.right, env)

        if node.op == "+":
            return left + right
        if node.op == "-":
            return left - right
        if node.op == "*":
            return left * right
        if node.op == "/":
            return left / right

        raise ValueError(f"Unknown operator: {node.op}")

    if isinstance(node, Variable):
        return env.get(node.name)

    if isinstance(node, Assignment):
        value = evaluate(node.value, env)
        env.define(node.name, value)
        return value

    if isinstance(node, Comparison):
        left = evaluate(node.left, env)
        right = evaluate(node.right, env)

        if node.op == ">":
            return left > right
        if node.op == "<":
            return left < right
        if node.op == "==":
            return left == right

    raise TypeError(f"Unknown node: {node}")

In [79]:
env = Environment()
evaluate(Assignment("x", Number(5)), env)

print(evaluate(Comparison(">", Variable("x"), Number(3)), env))
print(evaluate(Comparison("<", Variable("x"), Number(3)), env))
print(evaluate(Comparison("==", Variable("x"), Number(5)), env))

True
False
True


# Part 3 — Live Coding: Add IfStatement

Prompt:

> What should the evaluator do first when it sees an `IfStatement`?

Expected:

```text
Evaluate the condition.
```

In [80]:
# Starter: add the IfStatement case

def evaluate_if(node, env):
    # Reuse the complete comparison evaluator for old cases.
    if isinstance(node, (Number, Boolean, BinaryOp, Variable, Assignment, Comparison)):
        return evaluate(node, env)

    if isinstance(node, IfStatement):
        if evaluate_if(node.condition, env):
            evaluate_if(node.body, env)
        return None


    raise TypeError(f"Unknown node: {node}")

In [81]:
# True branch

env = Environment()
evaluate_if(Assignment("x", Number(5)), env)

program = IfStatement(
    condition=Comparison(">", Variable("x"), Number(3)),
    body=Assignment("y", Number(10)),
)

print("result:", evaluate_if(program, env))
print("env:", env)

result: None
env: Environment({'x': 5, 'y': 10})


In [82]:
# False branch

env = Environment()
evaluate_if(Assignment("x", Number(1)), env)

program = IfStatement(
    condition=Comparison(">", Variable("x"), Number(3)),
    body=Assignment("y", Number(10)),
)

print("result:", evaluate_if(program, env))
print("env:", env)

result: None
env: Environment({'x': 1})


# Part 4 — Live Coding: Add WhileStatement

Key phrase:

```text
A while loop is an if statement that keeps asking the same question.
```

In [83]:
# Starter: add the WhileStatement case

def evaluate_while(node, env):
    if isinstance(node, (Number, Boolean, BinaryOp, Variable, Assignment, Comparison)):
        return evaluate(node, env)

    if isinstance(node, IfStatement):
        return evaluate_if(node, env)

    # TODO: handle while
    if isinstance(node, WhileStatement):
        while evaluate_while(node.condition, env):
            evaluate_while(node.body, env)
        return None


    raise TypeError(f"Unknown node: {node}")

In [84]:
env = Environment()
evaluate_while(Assignment("x", Number(0)), env)

loop = WhileStatement(
    condition=Comparison("<", Variable("x"), Number(3)),
    body=Assignment(
        "x",
        BinaryOp("+", Variable("x"), Number(1))
    ),
)

result = evaluate_while(loop, env)

print("result:", result)
print("final env:", env)

result: None
final env: Environment({'x': 3})


In [ ]:
now lets extend evaluate

In [66]:

def evaluate(node, env):
    if isinstance(node, Number):
        return node.value

    if isinstance(node, Boolean):
        return node.value

    if isinstance(node, BinaryOp):
        left = evaluate(node.left, env)
        right = evaluate(node.right, env)

        if node.op == "+":
            return left + right
        if node.op == "-":
            return left - right
        if node.op == "*":
            return left * right
        if node.op == "/":
            return left / right

        raise ValueError(f"Unknown operator: {node.op}")

    if isinstance(node, Variable):
        return env.get(node.name)

    if isinstance(node, Assignment):
        value = evaluate(node.value, env)
        env.define(node.name, value)
        return value

    # TODO:
    if isinstance(node, Comparison):
        left = evaluate(node.left, env)
        right = evaluate (node.right,env)
        if node.op == ">":
            return left > right
        if node.op == "<":
            return left< right
        if node.op == "==":
            return left == right

    if isinstance(node, IfStatement):
        if evaluate(node.condition, env):
            evaluate(node.body, env)
        return None

    if isinstance(node, WhileStatement):
        while evaluate(node.condition, env):
            evaluate(node.body, env)
        return None

    
    raise TypeError(f"Unknown node: {node}")

# Part 5 — Combined Example

This manually builds an AST for:

```python
x = 0
while x < 5:
    x = x + 1
if x == 5:
    y = 100
```

In [67]:
env = Environment()

stmt1 = Assignment("x", Number(0))

stmt2 = WhileStatement(
    condition=Comparison("<", Variable("x"), Number(5)),
    body=Assignment(
        "x",
        BinaryOp("+", Variable("x"), Number(1))
    ),
)

stmt3 = IfStatement(
    condition=Comparison("==", Variable("x"), Number(5)),
    body=Assignment("y", Number(100)),
)

evaluate(stmt1, env)
evaluate(stmt2, env)
evaluate(stmt3, env)

env

Environment({'x': 5, 'y': 100})

# Key Takeaway

Every control-flow feature follows the same pattern:

```text
New syntax
↓
New AST node
↓
New evaluation rule
↓
Tests
```

For `if`:

```text
evaluate condition once
execute body only if true
```

For `while`:

```text
evaluate condition repeatedly
execute body until condition becomes false
```